In [ ]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

# Add both the repo root and src so imports work for scripts/ and hotelling/
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

In [ ]:
from hotelling.spatial.census import build_grid_polygons

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Pop grid was saved with point geometry (midpoints). Convert to 100m square polygons.
grid = build_grid_polygons(grid)
grid['index'] = grid.index
print(f"Grid: {len(grid)} cells as square polygons")


In [ ]:
stadtstruktur = gpd.read_parquet(PATH_PROCESSED / 'gebaeude_stadtstruktur.parquet')
print(f"Stadtstruktur: {len(stadtstruktur)} loaded with population and area data")

In [ ]:
from hotelling.spatial.city_data import download_fnp_data

download_fnp_data()

fnp = gpd.read_file(PATH_RAW / 'fnp_2025.gpkg')

In [ ]:
from hotelling.spatial.city_data import download_brw_data

download_brw_data()

brw = gpd.read_file(PATH_RAW / 'brw_2025.gpkg')

In [ ]:
from hotelling.spatial.osm import fetch_site_eligibility_signals

gdf_blockers, gdf_vacant = fetch_site_eligibility_signals()

In [ ]:
from hotelling.spatial.city_data import download_alkis_data

download_alkis_data()
alkis = gpd.read_file(PATH_RAW / 'alkis_full.gpkg')

In [ ]:
# FNP with only polygons
fnp_poly  = fnp[fnp.geometry.type.isin(['Polygon', 'MultiPolygon'])].copy()
print(f"FNP: {len(fnp)} total, {len(fnp_poly)} with polygon geometries")

In [ ]:
# Join FNP and BRW data into stadtstruktur by spatial join

stadtstruktur_full = stadtstruktur.copy()

stadtstruktur_full = gpd.sjoin(stadtstruktur_full, fnp_poly, how='left', predicate='intersects')
stadtstruktur_full.rename(columns={'index_right': 'index_fnp'}, inplace=True)



In [ ]:
# 1. ALKIS Tatsächliche Nutzung

alkis_TN = gpd.read_file(PATH_RAW / 'alkis_full.gpkg', layer='tatsaechlichenutzungflaechen').rename(columns={'id': 'id_TN', 'bezeich': 'bezeich_TN', 'uuid': 'uuid_TN'})
alkis_FS = gpd.read_file(PATH_RAW / 'alkis_full.gpkg', layer='flurstuecke').rename(columns={'id': 'id_FS'})
print(f"ALKIS TN: {len(alkis_TN)} features")
print(f"ALKIS FS: {len(alkis_FS)} features")

In [ ]:
'''# Extend FS by the TN columns with np.nan values for missing matches
alkis_FS_TN = alkis_FS.copy()
for col in alkis_TN.columns:
    if col not in alkis_FS_TN.columns:
        alkis_FS_TN[col] = np.nan
    else:
        print(f"Column {col} already exists in FS, skipping extension for this column.")'''

In [ ]:
alkis_FS_TN = gpd.sjoin(alkis_FS, alkis_TN, how='left', predicate='intersects')
print(f"Joined ALKIS FS with TN: {len(alkis_FS_TN)} features after spatial join")

In [ ]:
from scripts.spatial_helpers import area_coverage_fractions

cov_dict = area_coverage_fractions(left = alkis_FS, right = alkis_TN,
                                   left_id = "id_FS", right_id = "id_TN")

# Build mapping of fs_id -> best_id_TN (vectorized approach)
best_tn_map = {fs_id: max(coverage, key=coverage.get) 
               for fs_id, coverage in cov_dict.items()}

# Keep only rows where id_TN matches the best id_TN for that fs_id
alkis_FS_TN = alkis_FS_TN[
    alkis_FS_TN['id_FS'].isin(best_tn_map.keys()) &
    (alkis_FS_TN['id_TN'] == alkis_FS_TN['id_FS'].map(best_tn_map))
]

print(f"Filtered ALKIS FS-TN: {len(alkis_FS_TN)} rows (kept best id_TN per id_FS)")

In [ ]:
print(alkis_FS_TN['bezeich_TN'].unique())

# Remove bezeich_TN in ['AX_Weg', 'AX_Strassenverkehr', 'AX_SportFreizeitUndErholungsflaeche', 'AX_Landwirtschaft', 'AX_Hafenbecken', 'AX_StehendesGewaesser', 'AX_Gehoelz', 'AX_Wald', 'AX_Fliessgewaesser', 'AX_Friedhof', 'AX_Heide', 'AX_Schiffsverkehr', 'AX_Sumpf', 'AX_Halde', 'AX_Moor', 'AX_Flugverkehr', 'AX_TagebauGrubeSteinbruch']

alkis_FS_TN = alkis_FS_TN[~alkis_FS_TN['bezeich_TN'].isin([
    'AX_Weg', 'AX_Strassenverkehr', 'AX_SportFreizeitUndErholungsflaeche', 'AX_Landwirtschaft', 'AX_Hafenbecken', 'AX_StehendesGewaesser', 'AX_Gehoelz', 'AX_Wald', 'AX_Fliessgewaesser', 'AX_Friedhof', 'AX_Heide', 'AX_Schiffsverkehr', 'AX_Sumpf', 'AX_Halde', 'AX_Moor', 'AX_Flugverkehr', 'AX_TagebauGrubeSteinbruch'
])]

In [ ]:
alkis_FS_TN = alkis_FS_TN.rename(columns={'index_right': 'index_TN'})

In [ ]:
alkis_FS_TN_FNP = gpd.sjoin(alkis_FS_TN, fnp_poly, how='left', predicate='intersects')
print(f"Joined ALKIS FS TN with fnp_poly: {len(alkis_FS_TN_FNP)} features after spatial join")

In [ ]:
cov_dict = area_coverage_fractions(left = alkis_FS_TN, right = fnp_poly,
                                   left_id = "id_FS", right_id = "gisid")

# Build mapping of fs_id -> best_id_TN (vectorized approach)
best_fnp_map = {}
for fs_id, coverage in cov_dict.items():
    if len(coverage) == 0:
        print(f"id_FS {fs_id} has no coverage in FNP layer")
    else:
        best_fnp_map[fs_id] = max(coverage, key=coverage.get)

# Keep only rows where id_TN matches the best id_TN for that fs_id
alkis_FS_TN_FNP = alkis_FS_TN_FNP[
    alkis_FS_TN_FNP['id_FS'].isin(best_fnp_map.keys()) &
    (alkis_FS_TN_FNP['gisid'] == alkis_FS_TN_FNP['id_FS'].map(best_fnp_map))
]

print(f"Filtered ALKIS FS-TN: {len(alkis_FS_TN_FNP)} rows (kept best id_TN per id_FS)")

In [ ]:
alkis_FS_TN_FNP = alkis_FS_TN_FNP.rename(columns={'index_right': 'index_fnp'})
alkis_FS_TN_FNP

In [ ]:
gebaeude = gpd.read_file(PATH_RAW / 'alkis_full.gpkg', layer='gebaeudeflaechen')
gebaeude = gebaeude[gebaeude['bezeich'] == "AX_Gebaeude"]

all_gebaeude = gebaeude.copy()

# Leave only buildings that have area >600 m^2 (to small to facilitate a supermarket)
gebaeude = gebaeude[gebaeude.geometry.area > 600]
print(f"Gebaeude: {len(gebaeude)} buildings with area >600 m^2")

gebaeude['size_flag'] = gebaeude.geometry.area.apply(lambda x: 'small' if x <= 800 else 'standard')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL: GFK_REJECT (REPLACEMENT — replaces the existing GFK_REJECT cell)
# ══════════════════════════════════════════════════════════════════════════════
#
# Expanded from the original "100% certainty" criterion to reflect empirical
# observation: buildings visible in the map output that are demonstrably
# unsuitable. All added codes represent operationally mono-functional buildings
# with restricted public access or institutionally controlled ground floors
# where inserting a supermarket tenancy is not legally or practically possible.
 
GFK_REJECT: set[int] = {
 
    # ── Rural / agricultural / forestry ───────────────────────────────────────
    1025, 1210, 1220, 1223,
    1311, 1312, 1313,
    2074,
    2700, 2720, 2721, 2723, 2724, 2726, 2727, 2735, 2741, 2742,
 
    # ── Too small / structural impossibility ──────────────────────────────────
    2055,   # Kiosk
    2211, 2212, 2213,   # Windmühle, Wassermühle, Schöpfwerk
 
    # ── Trade fair halls ──────────────────────────────────────────────────────
    2060,   # Messehalle — single-purpose, typically 10 000+ m², outside Ring anyway
 
    # ── Utility / supply infrastructure (Versorgung) ──────────────────────────
    2500, 2510, 2511, 2512, 2513,
    2520, 2521, 2522, 2523,
    2540, 2560, 2571, 2580, 2591,
 
    # ── Waste / disposal infrastructure (Entsorgung) ──────────────────────────
    2600, 2610, 2611, 2612,
    2620, 2621, 2622, 2623,
 
    # ── Transport operational (non-passenger-facing) ──────────────────────────
    2130,                       # Tankstelle
    2410, 2411, 2412,           # Straßenverkehr operational
    2420, 2421, 2423,           # Schienenverkehr operational
    2430, 2431,                 # Flugverkehr
    2440, 2441, 2442, 2443, 2444,  # Schiffsverkehr
 
    # ── Parking (non-building or underground) ─────────────────────────────────
    2462, 2463, 2465,
 
    # ── Government and administrative buildings ───────────────────────────────
    # ADDED: observed in output — Reichstag (3010), Rotes Rathaus (3012).
    # These carry permanent institutional occupancy with controlled access;
    # no ground-floor commercial tenancy is legally permissible.
    # NOTE: 3013 Post is deliberately KEPT — former post office buildings
    # (Postamt) are active conversion targets for retail in Berlin.
    3010,   # Verwaltungsgebäude (generic admin building)
    3011,   # Parlament
    3012,   # Rathaus
    3014,   # Zollamt
    3015,   # Gericht
    3016,   # Botschaft / Konsulat
 
    # ── Education and research ────────────────────────────────────────────────
    # ADDED: observed in output — HU Berlin Hauptgebäude (3023), schools (3021/3022).
    3020,   # Bildung und Forschung allgemein
    3021,   # Allgemeinbildende Schule
    3022,   # Berufsbildende Schule
    3023,   # Hochschulgebäude
    3024,   # Forschungsinstitut
 
    # ── Cultural institutions ─────────────────────────────────────────────────
    # ADDED: observed — Altes Museum, Bodemuseum (3034).
    # Protected Kulturgut; ground floors are exhibition space, not leasable units.
    # NOTE: 3013 Post kept (see above). 3091 Bahnhofsgebäude kept — station
    # buildings are the canonical Bahnhofs-Rewe/Netto locations.
    3030,   # Kulturelle Zwecke allgemein
    3032,   # Theater / Oper
    3033,   # Konzertgebäude
    3034,   # Museum
    3035,   # Rundfunk / Fernsehen
    3036,   # Veranstaltungsgebäude
    3037,   # Bibliothek / Bücherei
 
    # ── Religious buildings ───────────────────────────────────────────────────
    3040, 3041, 3042, 3043, 3044, 3045, 3046,
 
    # ── Healthcare ────────────────────────────────────────────────────────────
    # ADDED: observed — hospitals in output.
    # Hospital ground floors are clinical / regulated environments.
    3050,   # Gesundheitswesen allgemein
    3051,   # Krankenhaus
    3052,   # Heilanstalt / Pflegeanstalt
 
    # ── Social welfare ────────────────────────────────────────────────────────
    # ADDED: youth homes, senior centres, homeless shelters.
    # Institutionally managed; no commercial sub-letting of ground floor.
    3060, 3061, 3062, 3063, 3064, 3065,
 
    # ── Security / military ───────────────────────────────────────────────────
    3071, 3072, 3073, 3074, 3075,
 
    # ── Cemetery buildings ────────────────────────────────────────────────────
    3080, 3081, 3082,
 
    # ── Airport ───────────────────────────────────────────────────────────────
    3092,
 
    # ── Sport and leisure ─────────────────────────────────────────────────────
    # ADDED: sport halls visible in output.
    3210,   # Sportzwecke allgemein
    3211,   # Sport- / Turnhalle
    3212,   # Gebäude zum Sportplatz
    3220,   # Badegebäude allgemein
    3221,   # Hallenbad
    3222,   # Gebäude im Freibad
    3240,   # Kurbetrieb
    3241,   # Badegebäude medizinisch
    3242,   # Sanatorium
 
    # ── Zoo / botanical garden ────────────────────────────────────────────────
    3260, 3261, 3262, 3263, 3264,
    3270, 3271, 3272, 3273,
 
    # ── Rural shelter / historic fortification ────────────────────────────────
    3281, 3031, 3038,
}
 
gebaeude["gfk"] = gebaeude["gfk"].fillna(0).astype(int)
gebaeude = gebaeude[~gebaeude["gfk"].isin(GFK_REJECT)]
print(f"After GFK_REJECT filter: {len(gebaeude)} buildings remain")

In [ ]:
buildings_candidate = gpd.sjoin(gebaeude, alkis_FS_TN_FNP, how='left', predicate='intersects')

cov_dict = area_coverage_fractions(left = gebaeude, right = alkis_FS_TN_FNP,
                                   left_id = "id", right_id = "id_FS")

best_alkis_map = {}
for building_id, coverage in cov_dict.items():
    if len(coverage) == 0:
        print(f"Building id {building_id} has no coverage in ALKIS FS layer")
    else:
        best_alkis_map[building_id] = max(coverage, key=coverage.get)

buildings_candidate = buildings_candidate[
    buildings_candidate['id'].isin(best_alkis_map.keys()) &
    (buildings_candidate['id_FS'] == buildings_candidate['id'].map(best_alkis_map))
]

print(f"Buildings candidate set after ALKIS FS-TN-FNP join and best match filtering: {len(buildings_candidate)} buildings")

In [ ]:
OSM_supermarkets = gpd.read_parquet(PATH_PROCESSED / 'supermarkets.parquet')
OSM_supermarkets.geometry = OSM_supermarkets.geometry.buffer(25)  # Buffer by 25m to account for geocoding imprecision

buildings_candidate['contains_supermarket'] = buildings_candidate.geometry.apply(
    lambda geom: OSM_supermarkets.to_crs(buildings_candidate.crs).geometry.intersects(geom).any()
)

buildings_candidate['in_grid'] = buildings_candidate.to_crs(grid.crs).within(grid.unary_union)

In [ ]:
grid_building_candidates = buildings_candidate[buildings_candidate['in_grid']].copy()
grid_building_candidates = grid_building_candidates[~grid_building_candidates['contains_supermarket']]

In [ ]:
OSM_groud_floor_blockers = gpd.read_file(PATH_RAW / 'OSM_site_eligibility_Berlin.gpkg', layer='blockers')
OSM_vacant = gpd.read_file(PATH_RAW / 'OSM_site_eligibility_Berlin.gpkg', layer='vacant')

In [ ]:
OSM_groud_floor_blockers = OSM_groud_floor_blockers.to_crs(grid.crs)
OSM_vacant = OSM_vacant.to_crs(grid.crs)

In [ ]:
# OSM_groud_floor_blockers = OSM_groud_floor_blockers[OSM_groud_floor_blockers.within(grid.unary_union)]
# OSM_groud_floor_blockers[OSM_groud_floor_blockers['osm_type'] == 'node'].geometry = OSM_groud_floor_blockers[OSM_groud_floor_blockers['osm_type'] == 'node'].geometry.buffer(25)

OSM_groud_floor_blockers = OSM_groud_floor_blockers[OSM_groud_floor_blockers.within(grid.union_all())]
mask = OSM_groud_floor_blockers['osm_type'] == 'node'
OSM_groud_floor_blockers.loc[mask, 'geometry'] = OSM_groud_floor_blockers.loc[mask, 'geometry'].buffer(25)

In [ ]:
OSM_groud_floor_blockers['amenity'].unique()

In [ ]:
from scripts.spatial_helpers import multi_sjoin

blockers_mapping = multi_sjoin(grid_building_candidates, OSM_groud_floor_blockers.to_crs(grid_building_candidates.crs), left_id= 'id', right_id='osm_id', predicate='intersects')
blockers_mapping_keys = blockers_mapping.keys()
grid_building_candidates['has_blocker'] = grid_building_candidates['id'].apply(lambda x: (x in blockers_mapping_keys) and (len(blockers_mapping[x]) > 0))
grid_building_candidates['n_blockers'] = grid_building_candidates['id'].apply(lambda x: len(set(blockers_mapping[x])) if x in blockers_mapping_keys else 0)

In [ ]:
grid_building_candidates

In [ ]:
print("=" * 60)
print("BLOCKERS")
print("  columns:", OSM_groud_floor_blockers.columns.tolist())
print("  geom_type counts:",
      OSM_groud_floor_blockers.geometry.geom_type.value_counts().to_dict())
print("  blocker_category counts:",
      OSM_groud_floor_blockers["blocker_category"].value_counts().to_dict())
print()
print("VACANT")
print("  columns:", OSM_vacant.columns.tolist())
print("  geom_type counts:",
      OSM_vacant.geometry.geom_type.value_counts().to_dict())
print("  vacant_category counts:",
      OSM_vacant["vacant_category"].value_counts().to_dict())
print("  'building' values:",
      OSM_vacant["building"].value_counts().to_dict()
      if "building" in OSM_vacant.columns else "column absent")
print()
print("BRW")
print("  columns:", brw.columns.tolist())
print("  CRS:", brw.crs)
print()
print("STADTSTRUKTUR (gebaeude_stadtstruktur.parquet)")
print("  columns:", stadtstruktur.columns.tolist())
print("  CRS:", stadtstruktur.crs)
print()
print("GRID BUILDING CANDIDATES")
print("  columns:", grid_building_candidates.columns.tolist())
print("  CRS:", grid_building_candidates.crs)
print(f"  count: {len(grid_building_candidates)}")

In [ ]:
from scripts.spatial_helpers import compute_ground_floor_availability
 
ANALYSIS_CRS = "EPSG:3035"
MIN_SUPERMARKET_M2 = 400.0        # minimum viable supermarket footprint
 
# ── AFTER running D-1, set these two column names from the diagnostic output ──
# BRW value column (typically "brw" in Berlin WFS layer brw2025:brw_2025_vector)
BRW_VALUE_COL = "brw"             # ← adjust if diagnostic shows different name
 
# Stadtstruktur type column.  The b_stadtstruktur_differenziert_2024 layer
# typically carries "typ" (integer code) or "vbez" / "kurzbez" (text label).
# Run D-1 first, then set one of: "typ", "vbez", "kurzbez", "beschreibung"
SS_TYPE_COL    = "typ_klar"   # primary: fine-grained morphological type
SS_TYPE_BROAD  = "ststrname"  # fallback: coarser functional category
 

In [ ]:
avail_df = compute_ground_floor_availability(
    buildings         = grid_building_candidates,
    blockers          = OSM_groud_floor_blockers,
    building_id_col   = "id",
    blocker_cat_col   = "blocker_category",
    min_supermarket_area_m2 = MIN_SUPERMARKET_M2,
    analysis_crs      = ANALYSIS_CRS,
)
 
print(f"Availability computed for {len(avail_df)} buildings")
print(f"  is_blocked=True  : {avail_df['is_blocked'].sum()}")
print(f"  is_blocked=False : {(~avail_df['is_blocked']).sum()}")
print(f"  available_m2 summary:")
print(avail_df["available_m2"].describe().round(1))

In [ ]:
gbc = grid_building_candidates.merge(avail_df, on="id", how="left")
 
# Buildings with no blocker data at all → treat as fully available
gbc["footprint_m2"]     = gbc["footprint_m2"].fillna(gbc.geometry.area)
gbc["available_m2"]     = gbc["available_m2"].fillna(gbc["footprint_m2"])
gbc["is_blocked"]       = gbc["is_blocked"].fillna(False)
gbc["n_poly_blockers"]  = gbc["n_poly_blockers"].fillna(0).astype(int)
gbc["n_node_blockers"]  = gbc["n_node_blockers"].fillna(0).astype(int)
 
# Free set: buildings where ground floor has room for a supermarket
gbc_free = gbc[~gbc["is_blocked"]].copy()
print(f"Free buildings (available_m2 >= {MIN_SUPERMARKET_M2} m²): {len(gbc_free)}")

In [ ]:
import numpy as np
from shapely import STRtree
 
TARGET_CRS = ANALYSIS_CRS
 
bld_3035  = gbc_free.to_crs(TARGET_CRS)
vac_3035  = OSM_vacant.to_crs(TARGET_CRS).copy()
 
# Supermarket-specific vacants: building tag explicitly says supermarket
vac_super = vac_3035[
    vac_3035["building"].isin(["supermarket"])
] if "building" in vac_3035.columns else vac_3035.iloc[0:0]
 
# Former-retail vacants: any former shop or commercial building
vac_retail = vac_3035[
    vac_3035["vacant_category"].isin(["former_shop", "commercial_building", "vacant_unit"])
]
 
def _flag_any_match(bld_gdf: gpd.GeoDataFrame,
                    right_gdf: gpd.GeoDataFrame,
                    flag_col: str) -> pd.Series:
    """Return boolean Series indexed like bld_gdf: True if any right geometry
    intersects the building polygon (or is contained within it for points)."""
    if right_gdf.empty:
        return pd.Series(False, index=bld_gdf.index)
 
    bld_geoms = bld_gdf.geometry.to_numpy()
    right_geoms = right_gdf.geometry.to_numpy()
 
    # Separate point vs. polygon right-side geometries
    is_poly = np.array([g.geom_type in ("Polygon", "MultiPolygon")
                        for g in right_geoms])
    is_pt   = ~is_poly
 
    matched_bld_indices: set[int] = set()
 
    if is_poly.any():
        tree = STRtree(bld_geoms)
        r_idx, b_idx = tree.query(right_geoms[is_poly], predicate="intersects")
        matched_bld_indices.update(b_idx.tolist())
 
    if is_pt.any():
        tree = STRtree(bld_geoms)
        r_idx, b_idx = tree.query(right_geoms[is_pt], predicate="intersects")
        matched_bld_indices.update(b_idx.tolist())
 
    flag = pd.Series(False, index=bld_gdf.index)
    flag.iloc[list(matched_bld_indices)] = True
    return flag
 
gbc_free["was_supermarket"] = _flag_any_match(bld_3035, vac_super, "was_supermarket").values
gbc_free["was_retail"]      = _flag_any_match(bld_3035, vac_retail, "was_retail").values
 
print(f"was_supermarket=True : {gbc_free['was_supermarket'].sum()}")
print(f"was_retail=True      : {gbc_free['was_retail'].sum()}")

In [ ]:
brw_3035 = brw.to_crs(TARGET_CRS).copy()

# Auto-detect BRW value column if the guess was wrong
if BRW_VALUE_COL not in brw_3035.columns:
    numeric_cols = [c for c in brw_3035.select_dtypes(include=[np.number]).columns
                    if c not in ("index", "fid")]
    BRW_VALUE_COL = numeric_cols[0] if numeric_cols else None
    print(f"BRW_VALUE_COL auto-detected as: {BRW_VALUE_COL!r}")
    print("  Available numeric cols:", brw_3035.select_dtypes(include=[np.number]).columns.tolist())

# Slim down and give the right side an explicit integer key column
brw_slim = (brw_3035[["geometry", BRW_VALUE_COL]].copy() if BRW_VALUE_COL
            else brw_3035[["geometry"]].copy())
brw_slim = brw_slim.reset_index(drop=True)          # clean 0-based RangeIndex
brw_slim["_brw_idx"] = brw_slim.index               # explicit column, same values

brw_cov = area_coverage_fractions(
    left     = gbc_free.to_crs(TARGET_CRS),
    right    = brw_slim,
    left_id  = "id",
    right_id = "_brw_idx",
)

best_brw_map = {
    bld_id: max(coverage, key=coverage.get)
    for bld_id, coverage in brw_cov.items()
    if coverage
}

def _lookup_brw(bld_id: object) -> float:
    brw_idx = best_brw_map.get(bld_id)
    if brw_idx is None or not BRW_VALUE_COL:
        return np.nan
    return float(brw_slim.loc[brw_idx, BRW_VALUE_COL])   # direct positional lookup

gbc_free["brw_value"] = gbc_free["id"].map(_lookup_brw)
print(f"BRW join: {gbc_free['brw_value'].notna().sum()}/{len(gbc_free)} buildings matched")
print(f"BRW range: {gbc_free['brw_value'].min():.0f} – {gbc_free['brw_value'].max():.0f} €/m²")

In [ ]:
# ── Score table: typ_klar (primary) ──────────────────────────────────────────
SS_SCORE_TYP_KLAR: dict[str, float] = {
 
    # ── Kerngebiet ────────────────────────────────────────────────────────────
    "Kerngebiet":                                                          0.95,
    # Purpose-designated city-centre commercial zone. Highest commercial density.
 
    # ── Gründerzeit block edge / perimeter block (1870s–1918) ─────────────────
    # These are the canonical inner-Berlin supermarket buildings.
    # Dense, 5–6 storeys, continuous street frontage, commercial GF standard.
    "Dichte Blockbebauung, geschlossener Hinterhof (1870er - 1918), 5 - 6-geschossig":  0.95,
    "Geschlossene Blockbebauung, Hinterhof (1870er - 1918), 5-geschossig":               0.93,
    "Geschlossene und halboffene Blockbebauung, Schmuck- und Gartenhof (1870er - 1918), 4-geschossig": 0.87,
    "Entkernte Blockrandbebauung, Lückenschluss nach 1945":                               0.72,
    # 'Entkernte' = Gründerzeit perimeter with interior removed post-war.
    # Street frontage buildings survive → still high GF commercial suitability.
 
    # ── Large-scale retail / commercial zones ────────────────────────────────
    # 'großflächiger Einzelhandel' explicitly labels large-footprint retail areas.
    "Gewerbe- und Industriegebiet, großflächiger Einzelhandel, dichte Bebauung":         0.88,
    "Gewerbe- und Industriegebiet, großflächiger Einzelhandel, geringe Bebauung":        0.80,
    # Note: 'geringe Bebauung' = sparse built coverage → lower density but
    # explicitly retail-zoned; still high priority.
 
    # ── Mixed-use without residential character ───────────────────────────────
    "Mischgebiet ohne Wohngebietscharakter, dichte Bebauung":                            0.80,
    "Mischgebiet ohne Wohngebietscharakter, geringe Bebauung":                           0.65,
 
    # ── Interwar block edge (1920s–1940s) ────────────────────────────────────
    # Rationalist/Neues Bauen blocks. Ground floor commercial less systematic
    # than Gründerzeit but still common in main shopping streets.
    "Blockrandbebauung mit Großhöfen (1920er - 1940er), 2 - 5-geschossig":              0.80,
    "Parallele Zeilenbebauung mit architektonischem Zeilengrün (1920er - 1930er), 2 - 5-geschossig": 0.52,
    # 'Parallel' = free-standing Zeilenbau → much less GF commercial than block edge.
 
    # ── Other mixed / inner-city ──────────────────────────────────────────────
    "Mischbebauung, halboffener und offener Schuppenhof, 2 - 4-geschossig":             0.68,
    # Older mixed development with workshop yards — transitional zone, good conversion.
    "Heterogene, innerstädtische Mischbebauung, Lückenschluss nach 1945":               0.65,
    # Post-war infill in otherwise Gründerzeit context — variable.
 
    # ── Station / rail areas ──────────────────────────────────────────────────
    "Bahnhof und Bahnanlagen ohne Gleiskörper":                                          0.82,
    # Station buildings and their surroundings. Bahnhofs-Rewe/Netto are a real
    # category. The 'ohne Gleiskörper' excludes track-side areas.
 
    # ── Administrative buildings ──────────────────────────────────────────────
    "Verwaltung":                                                                         0.50,
    # Admin buildings can have GF retail but it's not the norm.
 
    # ── Post-war residential (strip development) ─────────────────────────────
    "Freie Zeilenbebauung mit landschaftlichem Siedlungsgrün (1950er - 1970er), 2 - 6-geschossig": 0.32,
    # Free-standing Zeilenbau: designed as pure residential. Occasional GF
    # commercial units exist as retrofit (Kaufhalle in the Platte era) but rare.
 
    # ── Post-1990 housing ─────────────────────────────────────────────────────
    "Geschosswohnungsbau der 1990er Jahre und jünger":                                   0.38,
    # Newer multi-storey housing sometimes includes deliberate GF retail.
 
    # ── Large housing estates / Plattenbau ───────────────────────────────────
    "Großsiedlung und Punkthochhäuser (1960er - 1990er), 4 - 11-geschossig und mehr":   0.20,
    # Engineered as self-contained communities with planned Kaufhallen, but
    # those are purpose-built retail pavilions, not in these building GFKs.
 
    # ── Single / low-density residential ────────────────────────────────────
    "Freistehende Einfamilienhäuser mit Gärten":                                         0.12,
    "Reihen- und Doppelhäuser mit Gärten":                                               0.15,
    "Verdichtung im Einzelhausgebiet, Mischbebauung mit Garten und halbprivater Umgrünung (1870er bis heute)": 0.30,
    "Villen und Stadtvillen mit parkartigen Gärten (überwiegend 1870er - 1945)":         0.10,
    "Wochenendhaus- und kleingartenähnliches Gebiet":                                    0.05,
    "Dörfliche Mischbebauung":                                                           0.28,
 
    # ── Cultural / public / education ────────────────────────────────────────
    "Kultur":                                                                             0.42,
    "Hochschule und Forschung":                                                          0.32,
    "Krankenhaus":                                                                        0.32,
    "Neubau-Schule (Baujahr nach 1945)":                                                 0.18,
    "Altbau-Schule (Baujahr vor 1945)":                                                  0.22,
    "Kindertagesstätte":                                                                  0.12,
    "Sonstige Jugendeinrichtung":                                                         0.12,
    "Sonstiges und heterogenes Gemeinbedarfs- und Sondergebiet":                         0.28,
 
    # ── Sport / leisure ───────────────────────────────────────────────────────
    "Sportanlage, gedeckt":                                                               0.28,
    # Large covered sports halls → possible conversion, large footprint.
    "Sportanlage, ungedeckt":                                                             0.08,
    "Parkplatz":                                                                          0.18,
 
    # ── Functional infrastructure ─────────────────────────────────────────────
    "Ver- und Entsorgung":                                                                0.08,
    "Sicherheit und Ordnung":                                                             0.10,
    "Kirche":                                                                             0.08,
    # Already filtered by GFK; score 0.08 as safety net.
 
    # ── Brownfield / construction ────────────────────────────────────────────
    "Brachfläche":                                                                        0.40,
    # Brownfields inside the Ring are active conversion targets.
    "Baustelle":                                                                          0.38,
 
    # ── Open / green / non-urban ─────────────────────────────────────────────
    "Kleingartenanlage":                                                                  0.05,
    "Park / Grünfläche":                                                                  0.04,
    "Gleiskörper":                                                                        0.03,
    "Friedhof":                                                                           0.03,
    "sonstige Verkehrsfläche":                                                            0.04,
    "Stadtplatz / Promenade":                                                             0.28,
    "Wald":                                                                               0.02,
    "Gewässer":                                                                           0.02,
    "Campingplatz":                                                                       0.03,
    "Landwirtschaft":                                                                     0.03,
    "Baumschule / Gartenbau":                                                             0.05,
}
 
# ── Score table: ststrname (fallback) ────────────────────────────────────────
SS_SCORE_STSTRNAME: dict[str, float] = {
    # Gründerzeit block types — explicitly named: highest scores
    "Blockbebauung der Gründerzeit mit Seitenflügeln und Hinterhäusern":               0.95,
    "Blockrandbebauung der Gründerzeit mit geringem Anteil von Seiten- und Hintergebäuden": 0.88,
    "Blockrandbebauung der Gründerzeit mit massiven Veränderungen":                    0.76,
    # Explicitly dominated by retail/services
    "Bebauung mit überwiegender Nutzung durch Handel und Dienstleistung":              0.92,
    # Interwar mixed
    "Blockrand- und Zeilenbebauung der 1920er und 1930er Jahre":                       0.72,
    # Commercial/industrial
    "Dichte Bebauung mit überwiegender Nutzung durch Gewerbe und Industrie":           0.68,
    "Geringe Bebauung mit überwiegender Nutzung durch Gewerbe und Industrie":          0.60,
    # Public / special use
    "Bebauung mit überwiegender Nutzung durch Gemeinbedarf und Sondernutzung, Baustelle oder Verkehrsflächen ohne Straßenland": 0.28,
    # Post-war / recent housing
    "Siedlungsbebauung der 1990er Jahre und jünger":                                   0.36,
    "Zeilenbebauung seit den 1950er Jahren":                                           0.33,
    "Hohe Bebauung der Nachkriegszeit":                                                0.22,
    # Low density / gardens
    "Bebauung mit Gärten und halbprivater Umgrünung":                                  0.26,
    "Niedrige Bebauung mit Hausgärten":                                                0.16,
    "Villenbebauung mit parkartigen Gärten":                                           0.10,
    "Dörfliche Bebauung":                                                              0.24,
    # Non-built / green
    "Nicht oder gering bebaute Flächen der Gemeinbedarfs- und Sondernutzungen sowie Grün- und Freiflächen": 0.06,
    "Gewässer":                                                                        0.02,
}
 
_DEFAULT_SS_SCORE = 0.50   # genuinely unknown type
 
def _morphology_score(typ_klar_val: object, ststrname_val: object) -> float:
    """Return morphology_score from typ_klar (primary) with ststrname fallback."""
    # Primary: typ_klar
    if typ_klar_val is not None and not (isinstance(typ_klar_val, float) and np.isnan(typ_klar_val)):
        s = str(typ_klar_val).strip()
        if s in SS_SCORE_TYP_KLAR:
            return SS_SCORE_TYP_KLAR[s]
 
    # Fallback: ststrname
    if ststrname_val is not None and not (isinstance(ststrname_val, float) and np.isnan(ststrname_val)):
        s = str(ststrname_val).strip()
        if s in SS_SCORE_STSTRNAME:
            return SS_SCORE_STSTRNAME[s]
 
    return _DEFAULT_SS_SCORE
 
# ── Spatial join buildings → Stadtstruktur ───────────────────────────────────
# Use the gebaeude_stadtstruktur GeoDataFrame as a polygon reference for the
# Stadtstruktur zones (its geometry IS the building footprint; since Stadtstruktur
# polygons are block-level, each zone is represented by multiple buildings sharing
# the same typ_klar/ststrname. Using area_coverage_fractions ensures we pick the
# dominant zone for each candidate building.)
 
_ss_cols_needed = [c for c in ["typ_klar", "ststrname", "geometry"]
                   if c in stadtstruktur.columns]
 
if "typ_klar" not in stadtstruktur.columns:
    print("ERROR: 'typ_klar' not in stadtstruktur columns.")
    print("Available:", stadtstruktur.columns.tolist())
else:
    ss_poly = stadtstruktur[_ss_cols_needed].to_crs(TARGET_CRS).copy()
    ss_poly = ss_poly.reset_index(drop=False).rename(columns={"index": "_ss_idx"})
 
    ss_cov = area_coverage_fractions(
        left     = gbc_free.to_crs(TARGET_CRS),
        right    = ss_poly,
        left_id  = "id",
        right_id = "_ss_idx",
    )
    best_ss_map = {
        bld_id: max(cov, key=cov.get)
        for bld_id, cov in ss_cov.items() if cov
    }
 
    def _lookup_ss_row(bld_id: object) -> tuple[object, object]:
        ss_idx = best_ss_map.get(bld_id)
        if ss_idx is None:
            return (None, None)
        row = ss_poly[ss_poly["_ss_idx"] == ss_idx]
        if len(row) == 0:
            return (None, None)
        r = row.iloc[0]
        return (
            r.get("typ_klar") if "typ_klar" in r.index else None,
            r.get("ststrname") if "ststrname" in r.index else None,
        )
 
    ss_attrs = gbc_free["id"].map(_lookup_ss_row)
    gbc_free["typ_klar"]   = ss_attrs.map(lambda t: t[0])
    gbc_free["ststrname"]  = ss_attrs.map(lambda t: t[1])
 
    gbc_free["morphology_score"] = gbc_free.apply(
        lambda r: _morphology_score(r["typ_klar"], r["ststrname"]),
        axis=1,
    )
 
    # ── Diagnostic summary ───────────────────────────────────────────────────
    print(f"Morphology score stats:")
    print(gbc_free["morphology_score"].describe().round(3))
    print(f"\ntyp_klar distribution (top 15):")
    print(gbc_free["typ_klar"].value_counts().head(15))
    print(f"\nststrname distribution:")
    print(gbc_free["ststrname"].value_counts())
    print(f"\nBuildings with score >= 0.70 (Tier B threshold): "
          f"{(gbc_free['morphology_score'] >= 0.70).sum()}")
    print(f"Buildings with score >= 0.85 (high confidence):   "
          f"{(gbc_free['morphology_score'] >= 0.85).sum()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL D-7b (COMPLETE REPLACEMENT)
# Replaces the previous D-7b in full.
# ══════════════════════════════════════════════════════════════════════════════
 
import shapely as _shp
import numpy as np
 
# ── MBR dimension helpers (defined here, reused in D-9c) ─────────────────────
def _mbr_sides(geom):
    """Return (min_side_m, max_side_m) of a geometry's minimum bounding rectangle."""
    try:
        mbr    = _shp.minimum_rotated_rectangle(geom)
        coords = np.array(mbr.exterior.coords)          # (5, 2) closed ring
        sides  = np.array([
            np.linalg.norm(coords[1] - coords[0]),
            np.linalg.norm(coords[2] - coords[1]),
            np.linalg.norm(coords[3] - coords[2]),
            np.linalg.norm(coords[0] - coords[3]),
        ])
        return float(sides.min()), float(sides.max())
    except Exception:
        return 0.0, 0.0
 
 
# ── 1. typ_klar hard exclusion ────────────────────────────────────────────────
TYP_KLAR_REJECT: set[str] = {
    # Schools
    "Neubau-Schule (Baujahr nach 1945)",
    "Altbau-Schule (Baujahr vor 1945)",
    "Hochschule und Forschung",
    "Kindertagesstätte",
    "Sonstige Jugendeinrichtung",
    # Healthcare
    "Krankenhaus",
    # Security / order
    "Sicherheit und Ordnung",
    # Utility
    "Ver- und Entsorgung",
    # Outdoor sport
    "Sportanlage, ungedeckt",
    # Religion
    "Kirche",
    # Government / cultural / civic — excluded in full: not evaluable case-by-case
    "Verwaltung",
    "Kultur",
    "Stadtplatz / Promenade",
    # Low-density residential
    "Freistehende Einfamilienhäuser mit Gärten",
    "Reihen- und Doppelhäuser mit Gärten",
    "Villen und Stadtvillen mit parkartigen Gärten (überwiegend 1870er - 1945)",
    "Wochenendhaus- und kleingartenähnliches Gebiet",
    # Open space / non-urban
    "Friedhof",
    "Gleiskörper",
    "sonstige Verkehrsfläche",
    "Gewässer",
    "Wald",
    "Landwirtschaft",
    "Baumschule / Gartenbau",
    "Campingplatz",
    "Park / Grünfläche",
    "Kleingartenanlage",
}
 
_before = len(gbc_free)
gbc_free = gbc_free[~gbc_free["typ_klar"].isin(TYP_KLAR_REJECT)].copy()
print(f"typ_klar hard exclusion:     removed {_before - len(gbc_free):>5}, "
      f"{len(gbc_free)} remain")
 
# ── 2. Active industrial zone filter ─────────────────────────────────────────
# Problem: buildings with GFKs 2100/2120/2143 etc. (kept for their conversion
# potential) appear inside active industrial complexes (Bayer, Siemens, etc.)
# where no public retail can operate.
#
# Rule: exclude buildings where BOTH conditions hold:
#   a) ststrname signals predominantly industrial land use
#   b) GFK confirms the building is functionally industrial
#
# This is more precise than ststrname alone (which would also drop legitimate
# large-format retail zones sharing the same ststrname) or GFK alone (which
# would drop warehouses being converted to lofts/retail elsewhere).
_ACTIVE_INDUSTRIAL_STSTRNAME: set[str] = {
    "Dichte Bebauung mit überwiegender Nutzung durch Gewerbe und Industrie",
    "Geringe Bebauung mit überwiegender Nutzung durch Gewerbe und Industrie",
}
# Functional industrial GFKs — operational buildings in a plant/factory complex.
# Deliberately excludes 2010 (Handel), 2020 (Büro), 2050 (Geschäft): standalone
# commercial buildings within an industrial zone can still host retail.
_ACTIVE_INDUSTRIAL_GFK: set[int] = {
    2100,   # Gewerbe und Industrie allgemein
    2111,   # Fabrik (also now in GFK_REJECT; belt-and-suspenders)
    2120,   # Werkstatt
    2140,   # Vorratshaltung allgemein
    2141,   # Kühlhaus
    2142,   # Speichergebäude
    2143,   # Lagerhalle / Lagerschuppen / Lagerhaus
    2150,   # Speditionsgebäude
    2160,   # Forschungszwecke
    2180,   # Betriebliche Sozialeinrichtung
    2200,   # Sonstiges Gewerbe und Industrie
}
 
_mask_industrial = (
    gbc_free["ststrname"].isin(_ACTIVE_INDUSTRIAL_STSTRNAME) &
    gbc_free["gfk"].isin(_ACTIVE_INDUSTRIAL_GFK)
)
_before = len(gbc_free)
gbc_free = gbc_free[~_mask_industrial].copy()
print(f"Active industrial zone excl.: removed {_before - len(gbc_free):>5}, "
      f"{len(gbc_free)} remain")
 
# ── 3. Plattenbau: exclude residential-GFK buildings ─────────────────────────
_RESIDENTIAL_GFK: set[int] = set(range(1000, 1200))
_PLATTENBAU_TYPE = (
    "Großsiedlung und Punkthochhäuser (1960er - 1990er), "
    "4 - 11-geschossig und mehr"
)
_mask_platte = (
    (gbc_free["typ_klar"] == _PLATTENBAU_TYPE) &
    (gbc_free["gfk"].isin(_RESIDENTIAL_GFK))
)
_before = len(gbc_free)
gbc_free = gbc_free[~_mask_platte].copy()
print(f"Plattenbau residential excl.: removed {_before - len(gbc_free):>5}, "
      f"{len(gbc_free)} remain")
 
# ── 4. Free Zeilenbebauung: raise minimum footprint ───────────────────────────
_ZEILEN_TYPE = (
    "Freie Zeilenbebauung mit landschaftlichem Siedlungsgrün "
    "(1950er - 1970er), 2 - 6-geschossig"
)
_mask_zeilen = (
    (gbc_free["typ_klar"] == _ZEILEN_TYPE) &
    (gbc_free["footprint_m2"] < 1_200.0)
)
_before = len(gbc_free)
gbc_free = gbc_free[~_mask_zeilen].copy()
print(f"Zeilen footprint < 1200 m²:   removed {_before - len(gbc_free):>5}, "
      f"{len(gbc_free)} remain")
 
# ── 5. Building MBR minimum dimension filter ──────────────────────────────────
# A building footprint narrower than 10 m in its minimum dimension cannot
# physically house a supermarket (checkout bank + one-way aisle = ~8 m minimum;
# 10 m gives marginal operational room).
# Computed directly on EPSG:25833 geometry (UTM metric, error < 0.5 % at Berlin).
_MIN_BUILDING_DIM_M = 10.0
 
_sides = [_mbr_sides(g) for g in gbc_free.geometry.values]
_min_dims = np.array([s[0] for s in _sides])
 
_before = len(gbc_free)
gbc_free = gbc_free[_min_dims >= _MIN_BUILDING_DIM_M].copy()
print(f"Building min dim < {_MIN_BUILDING_DIM_M:.0f} m:      removed {_before - len(gbc_free):>5}, "
      f"{len(gbc_free)} remain")
 
# ── Mark all surviving buildings as non-greenfield ───────────────────────────
gbc_free["is_greenfield"] = False
print(f"\nAfter D-7b: {len(gbc_free)} candidate buildings remain")

In [ ]:
TIER_B_MORPHOLOGY_THRESHOLD = 0.70
 
def _assign_tier(row: pd.Series) -> str:
    if row["was_supermarket"]:
        return "A"
    if row["was_retail"] or row.get("morphology_score", 0.0) >= TIER_B_MORPHOLOGY_THRESHOLD:
        return "B"
    return "C"
 
gbc_free["confidence_tier"] = gbc_free.apply(_assign_tier, axis=1)
 
tier_counts = gbc_free["confidence_tier"].value_counts()
print("Confidence tier distribution:")
print(tier_counts)
print(f"\nTotal candidate locations: {len(gbc_free)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL D-9 (REPLACEMENT — replaces existing D-9)
# Only change: add "is_greenfield" to _KEEP_NEW
# ══════════════════════════════════════════════════════════════════════════════
 
_KEEP_GBC = ["id", "geometry", "gfk", "size_flag"]
 
_KEEP_NEW = [
    "footprint_m2",
    "available_m2",
    "n_poly_blockers",
    "n_node_blockers",
    "is_blocked",
    "was_supermarket",
    "was_retail",
    "is_greenfield",       # ← new
    "brw_value",
    "typ_klar",
    "ststrname",
    "morphology_score",
    "confidence_tier",
]
 
_KEEP_ALKIS = [c for c in ["id_FS", "bezeich_TN", "gisid"]
               if c in gbc_free.columns]
 
keep_cols = _KEEP_GBC + _KEEP_ALKIS + _KEEP_NEW
keep_cols = [c for c in keep_cols if c in gbc_free.columns]
keep_cols = list(dict.fromkeys(keep_cols))
 
L_commercial_final = (
    gbc_free[keep_cols]
    .copy()
    .reset_index(drop=True)
)
L_commercial_final["centroid"] = (
    L_commercial_final.to_crs(TARGET_CRS).geometry.centroid
)
 
print(f"L_commercial_final (buildings only): {len(L_commercial_final)} locations")
print(f"  Tier A: {(L_commercial_final['confidence_tier']=='A').sum()}")
print(f"  Tier B: {(L_commercial_final['confidence_tier']=='B').sum()}")
print(f"  Tier C: {(L_commercial_final['confidence_tier']=='C').sum()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL D-9b (NEW — insert AFTER D-9, BEFORE D-10)
# Greenfield candidates: brownfields and large parking lots
# ══════════════════════════════════════════════════════════════════════════════
#
# Parallel track: plots where a NEW freestanding supermarket could be built.
# Uses raw Stadtstruktur zone polygons (not ALKIS buildings) as candidate
# geometry.
#
# INCLUDED:
#   Brachfläche  (brownfield) — no statutory protection, actively redeveloped
#                inside the Ring; min area 1 500 m² (smallest viable discounter
#                footprint + access)
#   Parkplatz    (surface parking) — large lots in commercial/mixed zones are
#                routinely converted; min area 2 500 m²; FNP must be
#                non-residential
#
# EXCLUDED (deliberately):
#   Kleingartenanlage — Berliner Kleingartenverordnung (2023) provides near-
#                statutory protection; only parcels designated Entwicklungsfläche
#                in FNP can be redeveloped; these are not identifiable without
#                full Bebauungsplan layer → too speculative, analytically
#                misleading to include
#   Baustelle    — already under active construction, future use determined
#   Open space / parks / cemeteries — legally protected
# ─────────────────────────────────────────────────────────────────────────────
 
import warnings
 
_GF_PARAMS: dict[str, dict] = {
    "Brachfläche": {"min_area_m2": 1_500.0, "morphology": 0.40},
    "Parkplatz":   {"min_area_m2": 2_500.0, "morphology": 0.22},
}
 
# FNP zone strings incompatible with retail development
_FNP_REJECT_SUBSTRINGS = ["WR", "GI", "GR", "PF", "AF", "FW"]
 
# ── Load raw Stadtstruktur zone polygons ──────────────────────────────────────
_ss_raw_path = PATH_RAW / "stadtstruktur.gpkg"
_ss_zones: gpd.GeoDataFrame | None = None
 
if _ss_raw_path.exists():
    _ss_zones = gpd.read_file(_ss_raw_path)
    print(f"Stadtstruktur zones: {len(_ss_zones)} polygons, "
          f"typ_klar present: {'typ_klar' in _ss_zones.columns}")
else:
    print(f"WARNING: {_ss_raw_path} not found.")
    print("Run download_stadtstruktur() to create it, then re-run this cell.")
 
if _ss_zones is not None and "typ_klar" in _ss_zones.columns:
 
    # ── Clip to inner-Ring boundary ───────────────────────────────────────────
    _ss_3035 = _ss_zones.to_crs(TARGET_CRS).copy()
    _bnd_3035 = boundary.to_crs(TARGET_CRS)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _ss_clipped = gpd.clip(_ss_3035, _bnd_3035).copy()
    _ss_clipped["_zone_area"] = _ss_clipped.geometry.area
    print(f"After clip to inner Ring: {len(_ss_clipped)} zone polygons")
 
    # ── FNP zone column auto-detection ────────────────────────────────────────
    _fnp_3035 = fnp_poly.to_crs(TARGET_CRS).copy()
    _fnp_zone_col: str | None = None
    for _c in ["nutzung", "NUTZUNG", "nutzung_bez", "NUTZUNG_BEZ",
               "nutzart", "NUTZART", "art", "ART"]:
        if _c in _fnp_3035.columns:
            _fnp_zone_col = _c
            break
    if _fnp_zone_col:
        print(f"FNP zone column: '{_fnp_zone_col}'  "
              f"(sample values: {_fnp_3035[_fnp_zone_col].dropna().unique()[:6].tolist()})")
    else:
        print("FNP zone column not found — FNP compatibility filter skipped.")
        print(f"  FNP columns: {_fnp_3035.columns.tolist()}")
 
    # ── BRW slim (reuse from D-6 if available, else rebuild) ─────────────────
    if "brw_slim" not in dir():
        _brw_for_gf = brw.to_crs(TARGET_CRS).copy().reset_index(drop=True)
        _brw_for_gf["_brw_idx"] = _brw_for_gf.index
    else:
        _brw_for_gf = brw_slim.to_crs(TARGET_CRS).copy() if hasattr(brw_slim, "crs") else brw_slim.copy()
 
    # ── Build greenfield records ──────────────────────────────────────────────
    _gf_records: list[dict] = []
 
    for _zone_name, _params in _GF_PARAMS.items():
        _zone_gdf = _ss_clipped[
            (_ss_clipped["typ_klar"] == _zone_name) &
            (_ss_clipped["_zone_area"] >= _params["min_area_m2"])
        ].copy().reset_index(drop=True)
 
        print(f"\n{_zone_name}: {len(_zone_gdf)} polygons "
              f">= {_params['min_area_m2']:.0f} m²")
 
        if _zone_gdf.empty:
            continue
 
        # ── FNP filter (sjoin → dominant zone → reject if WR/GI/GR etc.) ─────
        if _fnp_zone_col:
            _sj = gpd.sjoin(
                _zone_gdf[["geometry"]].reset_index(names="_gf_idx"),
                _fnp_3035[["geometry", _fnp_zone_col]],
                how="left",
                predicate="intersects",
            )
            # Dominant FNP zone per greenfield polygon = most frequent match
            _dom = (
                _sj.dropna(subset=[_fnp_zone_col])
                .groupby("_gf_idx")[_fnp_zone_col]
                .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
            )
            _zone_gdf["_fnp_zone"] = _zone_gdf.index.map(_dom)
 
            def _fnp_ok(val: object) -> bool:
                if val is None or (isinstance(val, float) and np.isnan(val)):
                    return True   # unknown → include (conservative)
                return not any(p in str(val) for p in _FNP_REJECT_SUBSTRINGS)
 
            _n_before = len(_zone_gdf)
            _zone_gdf = _zone_gdf[_zone_gdf["_fnp_zone"].map(_fnp_ok)].copy()
            print(f"  After FNP filter: {len(_zone_gdf)} "
                  f"(removed {_n_before - len(_zone_gdf)})")
        else:
            _zone_gdf["_fnp_zone"] = None
 
        # ── BRW lookup (bulk sjoin on centroid) ───────────────────────────────
        _centroids = _zone_gdf.copy()
        _centroids["geometry"] = _centroids.geometry.centroid
        _brw_sj = gpd.sjoin(
            _centroids[["geometry"]].reset_index(names="_gf_idx"),
            _brw_for_gf[["geometry", BRW_VALUE_COL]].reset_index(drop=True)
            if BRW_VALUE_COL else _brw_for_gf[["geometry"]].reset_index(drop=True),
            how="left",
            predicate="within",
        )
        _brw_lookup: dict[int, float] = {}
        if BRW_VALUE_COL and BRW_VALUE_COL in _brw_sj.columns:
            _brw_lookup = (
                _brw_sj.dropna(subset=[BRW_VALUE_COL])
                .groupby("_gf_idx")[BRW_VALUE_COL]
                .first()
                .to_dict()
            )
 
        # ── Assemble records ──────────────────────────────────────────────────
        _ststrname_col = "ststrname" if "ststrname" in _zone_gdf.columns else None
 
        for _i, _row in _zone_gdf.iterrows():
            _gf_records.append({
                "id":               f"GF_{_zone_name[:2].upper()}_{_i:04d}",
                "geometry":         _row.geometry,
                "gfk":              0,
                "size_flag":        ("standard"
                                     if _row["_zone_area"] >= 1_500
                                     else "small"),
                "footprint_m2":     round(_row["_zone_area"], 1),
                "available_m2":     round(_row["_zone_area"], 1),
                "n_poly_blockers":  0,
                "n_node_blockers":  0,
                "is_blocked":       False,
                "was_supermarket":  False,
                "was_retail":       False,
                "is_greenfield":    True,
                "brw_value":        _brw_lookup.get(_i, np.nan),
                "typ_klar":         _zone_name,
                "ststrname":        (_row[_ststrname_col]
                                     if _ststrname_col else None),
                "morphology_score": _params["morphology"],
                "confidence_tier":  "C",
            })
 
    # ── Concatenate into L_commercial_final ───────────────────────────────────
    if _gf_records:
        _gf_gdf = gpd.GeoDataFrame(
            _gf_records, geometry="geometry", crs=TARGET_CRS
        )
        _gf_gdf["centroid"] = _gf_gdf.geometry.centroid
 
        print(f"\nGreenfield candidates (Stadtstruktur zones): {len(_gf_gdf)}")
        print(f"  Brachfläche: "
              f"{sum(1 for r in _gf_records if r['typ_klar'] == 'Brachfläche')}")
        print(f"  Parkplatz:   "
              f"{sum(1 for r in _gf_records if r['typ_klar'] == 'Parkplatz')}")
 
        # ── CRS alignment before concat ───────────────────────────────────────
        # L_commercial_final is in ALKIS native CRS (EPSG:25833).
        # _gf_gdf is in TARGET_CRS (EPSG:3035).
        # Standardise everything on TARGET_CRS before merging.
        # Also drop the stale centroid column (computed in wrong CRS) and
        # recompute it after concat.
        _lcf_reproj = (
            L_commercial_final
            .drop(columns=["centroid"], errors="ignore")
            .to_crs(TARGET_CRS)
        )
 
        L_commercial_final = gpd.GeoDataFrame(
            pd.concat(
                [_lcf_reproj, _gf_gdf.drop(columns=["centroid"], errors="ignore")],
                ignore_index=True,
            ),
            geometry="geometry",
            crs=TARGET_CRS,
        )
        # Recompute centroid in the unified CRS
        L_commercial_final["centroid"] = L_commercial_final.geometry.centroid
 
        print(f"\nL_commercial_final after Stadtstruktur greenfield: "
              f"{len(L_commercial_final)} total")
    else:
        # Still reproject to TARGET_CRS for consistency with D-9c
        L_commercial_final = (
            L_commercial_final
            .drop(columns=["centroid"], errors="ignore")
            .to_crs(TARGET_CRS)
        )
        L_commercial_final["centroid"] = L_commercial_final.geometry.centroid
        print("No Stadtstruktur greenfield candidates found.")
 
else:
    # Stadtstruktur zones file unavailable: still reproject L_commercial_final
    L_commercial_final = (
        L_commercial_final
        .drop(columns=["centroid"], errors="ignore")
        .to_crs(TARGET_CRS)
    )
    L_commercial_final["centroid"] = L_commercial_final.geometry.centroid
    print("Greenfield track skipped (stadtstruktur.gpkg unavailable).")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL D-9c (COMPLETE REPLACEMENT)
# ══════════════════════════════════════════════════════════════════════════════
#
# Free-parcel track: cadastral parcels with no significant building coverage.
# Adds four constraints vs. the previous version:
#   1. MBR dimension filter — rejects extremely narrow or elongated parcels
#   2. Railway-specific dimensions — looser (arches can be longer) but road-
#      adjacent required (supermarket access needs a street alongside the track)
#   3. Road adjacency check for AX_Bahnverkehr using AX_Strassenverkehr areas
#   4. Same industrial-zone/road-adjacent logic applied consistently
# ─────────────────────────────────────────────────────────────────────────────
 
import shapely as _shp
import numpy as np
import warnings
 
# ── Constants ─────────────────────────────────────────────────────────────────
_TN_DEVELOPABLE: set[str] = {
    "AX_IndustrieUndGewerbeflaeche",
    "AX_GemischteNutzungsflaeche",
    "AX_FlaecheBesondererFunktionalerPraegung",
    "AX_Bahnverkehr",
}
_MIN_PARCEL_AREA_M2    = 1_500.0
_MAX_BUILDING_COV_FR   = 0.10
_FNP_REJECT_PATTERNS   = ["WR", "GI", "GR", "PF", "AF", "FW"]
 
# Dimension thresholds (applied in TARGET_CRS = EPSG:3035, metric)
_MIN_DIM_GENERAL  = 15.0    # m — minimum shorter side for non-rail parcels
_MAX_ASPECT_GEN   = 8.0     # max length:width ratio for non-rail parcels
_MIN_DIM_RAIL     = 10.0    # m — minimum shorter side for railway arches
_MAX_ASPECT_RAIL  = 25.0    # railway arches can be longer (multiple bays)
_ROAD_BUFFER_M    = 20.0    # m — rail parcel must have a road within this distance
 
# ── 1. Select developable parcels ─────────────────────────────────────────────
_parcels_dev = alkis_FS_TN_FNP[
    alkis_FS_TN_FNP["bezeich_TN"].isin(_TN_DEVELOPABLE)
].copy()
print(f"Developable TN parcels: {len(_parcels_dev)}")
print(f"  {_parcels_dev['bezeich_TN'].value_counts().to_dict()}")
 
_parcels_dev = _parcels_dev.to_crs(TARGET_CRS)
_bnd_3035    = boundary.to_crs(TARGET_CRS)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _parcels_dev = gpd.clip(_parcels_dev, _bnd_3035).copy()
_parcels_dev["_parcel_area"] = _parcels_dev.geometry.area
 
# ── 2. Minimum area filter ─────────────────────────────────────────────────────
_parcels_dev = _parcels_dev[
    _parcels_dev["_parcel_area"] >= _MIN_PARCEL_AREA_M2
].copy().reset_index(drop=True)
_parcels_dev["_pidx"] = _parcels_dev.index
print(f"After area ≥ {_MIN_PARCEL_AREA_M2:.0f} m²: {len(_parcels_dev)}")
 
# ── 3. MBR dimension filter ────────────────────────────────────────────────────
# Rejects parcels that are geometrically unsuitable regardless of area:
# a very long narrow strip cannot host a supermarket even at 2 000 m².
#
# Railway (AX_Bahnverkehr) arches have different thresholds: they are naturally
# elongated (many bays along a viaduct) but the SHORT dimension must still be
# ≥ 10 m (arch depth) and the length:width ratio is capped at 25:1.
 
_sides_p = [_mbr_sides(g) for g in _parcels_dev.geometry.values]
_parcels_dev["_min_dim"] = [s[0] for s in _sides_p]
_parcels_dev["_max_dim"] = [s[1] for s in _sides_p]
_parcels_dev["_aspect"]  = np.where(
    _parcels_dev["_min_dim"] > 0.01,
    _parcels_dev["_max_dim"] / _parcels_dev["_min_dim"],
    np.inf,
)
 
_is_rail = _parcels_dev["bezeich_TN"] == "AX_Bahnverkehr"
_dim_ok  = (
    (_is_rail  & (_parcels_dev["_min_dim"] >= _MIN_DIM_RAIL)    & (_parcels_dev["_aspect"] <= _MAX_ASPECT_RAIL))
    | (~_is_rail & (_parcels_dev["_min_dim"] >= _MIN_DIM_GENERAL) & (_parcels_dev["_aspect"] <= _MAX_ASPECT_GEN))
)
_before = len(_parcels_dev)
_parcels_dev = _parcels_dev[_dim_ok].copy()
print(f"MBR dimension filter: removed {_before - len(_parcels_dev)}, "
      f"{len(_parcels_dev)} remain "
      f"(rail: {(_parcels_dev['bezeich_TN']=='AX_Bahnverkehr').sum()}, "
      f"other: {(_parcels_dev['bezeich_TN']!='AX_Bahnverkehr').sum()})")
 
# ── 4. Road adjacency filter for AX_Bahnverkehr ───────────────────────────────
# Supermarkets under railway arches are only viable when a public road runs
# alongside — the viaduct itself is never demolished for retail access.
# Strategy: buffer each rail parcel by _ROAD_BUFFER_M and check intersection
# with AX_Strassenverkehr surfaces from ALKIS.
 
_rail_parcels = _parcels_dev[_parcels_dev["bezeich_TN"] == "AX_Bahnverkehr"].copy()
 
if len(_rail_parcels) > 0:
    # Load road surfaces (AX_Strassenverkehr) from the TN layer.
    # alkis_TN was loaded earlier with renamed columns (bezeich → bezeich_TN).
    if "alkis_TN" in dir() and "bezeich_TN" in alkis_TN.columns:
        _road_tn = alkis_TN[alkis_TN["bezeich_TN"] == "AX_Strassenverkehr"].copy()
    else:
        print("alkis_TN not in scope — reloading road layer from disk...")
        _road_raw = gpd.read_file(
            PATH_RAW / "alkis_full.gpkg",
            layer="tatsaechlichenutzungflaechen",
        )
        _road_tn = _road_raw[_road_raw["bezeich"] == "AX_Strassenverkehr"].copy()
 
    _road_ring = gpd.clip(
        _road_tn.to_crs(TARGET_CRS), _bnd_3035
    ).copy()
    print(f"Road surfaces in Ring: {len(_road_ring)}")
 
    # Buffer rail parcels and intersect with road surfaces
    _rail_buf = _rail_parcels[["_pidx", "geometry"]].copy()
    _rail_buf["geometry"] = _rail_buf.geometry.buffer(_ROAD_BUFFER_M)
 
    _road_sj = gpd.sjoin(
        _rail_buf,
        _road_ring[["geometry"]],
        how="inner",
        predicate="intersects",
    )
    _rail_with_road = set(_road_sj["_pidx"].unique())
    _rail_without   = set(_rail_parcels["_pidx"]) - _rail_with_road
 
    _before = len(_parcels_dev)
    _parcels_dev = _parcels_dev[
        ~_parcels_dev["_pidx"].isin(_rail_without)
    ].copy()
    print(f"Rail road-access filter: removed {len(_rail_without)}, "
          f"{len(_rail_with_road)} rail parcels kept (road within {_ROAD_BUFFER_M:.0f} m)")
else:
    print("No AX_Bahnverkehr parcels after dimension filter — road check skipped.")
 
# ── 5. Building coverage check ────────────────────────────────────────────────
if "all_gebaeude" not in dir():
    print("all_gebaeude not in scope — reloading...")
    _all_geb = gpd.read_file(
        PATH_RAW / "alkis_full.gpkg", layer="gebaeudeflaechen"
    )
    _all_geb = _all_geb[_all_geb["bezeich"] == "AX_Gebaeude"].copy()
else:
    _all_geb = all_gebaeude.copy()
 
_all_geb_ring = gpd.clip(
    _all_geb.to_crs(TARGET_CRS), _bnd_3035
).copy()
_all_geb_ring = _all_geb_ring[_all_geb_ring.geometry.area > 50].copy()
print(f"Buildings in Ring (coverage check): {len(_all_geb_ring)}")
 
_bld_sj = gpd.sjoin(
    _parcels_dev[["_pidx", "geometry"]],
    _all_geb_ring[["geometry"]].reset_index(names="_bidx"),
    how="left",
    predicate="intersects",
)
 
_pairs_clean = (
    _bld_sj.dropna(subset=["_bidx"])
    [["_pidx", "_bidx"]]
    .astype(int)
    .copy()
)
 
if len(_pairs_clean) > 0:
    _pg = _parcels_dev.set_index("_pidx")["geometry"]
    _bg = _all_geb_ring["geometry"]   # original index — matches _bidx values
 
    _p_arr = _pg.loc[_pairs_clean["_pidx"].values].values
    _b_arr = _bg.loc[_pairs_clean["_bidx"].values].values
 
    _inter_arr = _shp.area(_shp.intersection(_p_arr, _b_arr))
    _pairs_clean["_ia"] = _inter_arr
    _inter_sum = _pairs_clean.groupby("_pidx")["_ia"].sum()
else:
    _inter_sum = pd.Series(dtype=float)
 
_pa_map = _parcels_dev.set_index("_pidx")["_parcel_area"].clip(lower=1.0)
_parcels_dev["_bld_cov"] = (
    _inter_sum.reindex(_parcels_dev["_pidx"]).fillna(0.0).values
    / _pa_map.reindex(_parcels_dev["_pidx"]).values
)
 
_before = len(_parcels_dev)
_free_parcels = _parcels_dev[
    _parcels_dev["_bld_cov"] < _MAX_BUILDING_COV_FR
].copy()
print(f"Building coverage < {_MAX_BUILDING_COV_FR:.0%}: "
      f"{len(_free_parcels)} free parcels "
      f"(removed {_before - len(_free_parcels)} occupied)")
 
# ── 6. FNP compatibility check ────────────────────────────────────────────────
_fnp_zone_col_p: str | None = None
for _c in ["nutzung", "NUTZUNG", "nutzung_bez", "NUTZUNG_BEZ",
           "nutzart", "NUTZART", "art", "ART"]:
    if _c in _free_parcels.columns:
        _fnp_zone_col_p = _c
        break
 
if _fnp_zone_col_p:
    def _fnp_ok_p(val):
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return True
        return not any(p in str(val) for p in _FNP_REJECT_PATTERNS)
    _before = len(_free_parcels)
    _free_parcels = _free_parcels[_free_parcels[_fnp_zone_col_p].map(_fnp_ok_p)].copy()
    print(f"FNP filter: removed {_before - len(_free_parcels)}, "
          f"{len(_free_parcels)} remain")
else:
    print("FNP zone column not detected — skipped.")
    print(f"  Columns containing 'nutz'/'art': "
          f"{[c for c in _free_parcels.columns if any(x in c.lower() for x in ('nutz','art','fnp'))]}")
 
# ── 7. Stadtstruktur morphology score ─────────────────────────────────────────
_ss_raw_path_c = PATH_RAW / "stadtstruktur.gpkg"
if _ss_raw_path_c.exists():
    _ss_for_p = gpd.read_file(_ss_raw_path_c).to_crs(TARGET_CRS)
    _fp_cents = _free_parcels.copy()
    _fp_cents["geometry"] = _fp_cents.geometry.centroid
    _ss_cols = [c for c in ["geometry", "typ_klar", "ststrname"]
                if c in _ss_for_p.columns]
    _ss_sj = gpd.sjoin(
        _fp_cents[["_pidx", "geometry"]],
        _ss_for_p[_ss_cols],
        how="left",
        predicate="within",
    )
    _ss_lu = _ss_sj.drop_duplicates("_pidx").set_index("_pidx")
    _free_parcels["typ_klar_p"]  = _free_parcels["_pidx"].map(
        lambda i: _ss_lu["typ_klar"].get(i) if "typ_klar" in _ss_lu.columns else None
    )
    _free_parcels["ststrname_p"] = _free_parcels["_pidx"].map(
        lambda i: _ss_lu["ststrname"].get(i) if "ststrname" in _ss_lu.columns else None
    )
    _free_parcels["morphology_score_p"] = _free_parcels.apply(
        lambda r: _morphology_score(r.get("typ_klar_p"), r.get("ststrname_p")), axis=1
    )
else:
    _free_parcels["typ_klar_p"] = None
    _free_parcels["ststrname_p"] = None
    _free_parcels["morphology_score_p"] = 0.40
 
# ── 8. OSM vacant signal ──────────────────────────────────────────────────────
_vac_3035_p = OSM_vacant.to_crs(TARGET_CRS).copy()
_vac_fp = _vac_3035_p[
    _vac_3035_p["vacant_category"].isin(
        ["former_shop", "commercial_building", "vacant_unit"]
    )
].copy()
if not _vac_fp.empty:
    _fp_for_vac = _free_parcels.reset_index(drop=True).copy()
    _vac_sj = gpd.sjoin(
        _fp_for_vac[["geometry"]].reset_index(names="_pidx2"),
        _vac_fp[["geometry"]],
        how="left",
        predicate="intersects",
    )
    _was_retail_set = set(
        _vac_sj.dropna(subset=["index_right"])["_pidx2"].unique()
    )
    _free_parcels["was_retail_p"] = _free_parcels.index.isin(_was_retail_set)
else:
    _free_parcels["was_retail_p"] = False
 
# ── 9. BRW lookup ─────────────────────────────────────────────────────────────
if BRW_VALUE_COL and "brw_slim" in dir():
    _brw_for_p = brw_slim.to_crs(TARGET_CRS) if hasattr(brw_slim, "crs") else brw_slim
    _fp_cents2 = _free_parcels.copy()
    _fp_cents2["geometry"] = _fp_cents2.geometry.centroid
    _brw_sj_p = gpd.sjoin(
        _fp_cents2[["_pidx", "geometry"]],
        _brw_for_p[["geometry", BRW_VALUE_COL]],
        how="left",
        predicate="within",
    )
    _brw_p = (
        _brw_sj_p.dropna(subset=[BRW_VALUE_COL])
        .groupby("_pidx")[BRW_VALUE_COL].first()
        .to_dict()
    )
    _free_parcels["brw_value_p"] = _free_parcels["_pidx"].map(_brw_p)
else:
    _free_parcels["brw_value_p"] = np.nan
 
# ── 10. Assemble records and concat ───────────────────────────────────────────
_TN_MORPH_BASE = {
    "AX_IndustrieUndGewerbeflaeche":          0.55,
    "AX_GemischteNutzungsflaeche":             0.50,
    "AX_FlaecheBesondererFunktionalerPraegung": 0.45,
    "AX_Bahnverkehr":                          0.60,
}
_parcel_records: list[dict] = []
for _i, _row in _free_parcels.iterrows():
    _tn    = str(_row.get("bezeich_TN", ""))
    _morph = max(
        _TN_MORPH_BASE.get(_tn, 0.40),
        float(_row.get("morphology_score_p") or 0.0),
    )
    _parcel_records.append({
        "id":               f"FP_{_tn[-4:]}_{_i:05d}",
        "geometry":         _row.geometry,
        "gfk":              0,
        "size_flag":        "standard" if _row["_parcel_area"] >= 1_500 else "small",
        "footprint_m2":     round(_row["_parcel_area"], 1),
        "available_m2":     round(_row["_parcel_area"] * (1 - _row["_bld_cov"]), 1),
        "n_poly_blockers":  0,
        "n_node_blockers":  0,
        "is_blocked":       False,
        "was_supermarket":  False,
        "was_retail":       bool(_row.get("was_retail_p", False)),
        "is_greenfield":    True,
        "brw_value":        _row.get("brw_value_p", np.nan),
        "typ_klar":         _row.get("typ_klar_p"),
        "ststrname":        _row.get("ststrname_p"),
        "morphology_score": round(_morph, 3),
        "confidence_tier":  "B" if bool(_row.get("was_retail_p", False)) else "C",
        "id_FS":            _row.get("id_FS"),
        "bezeich_TN":       _tn,
    })
 
if _parcel_records:
    _fp_gdf = gpd.GeoDataFrame(
        _parcel_records, geometry="geometry", crs=TARGET_CRS
    )
    _lcf = L_commercial_final.drop(columns=["centroid"], errors="ignore")
    L_commercial_final = gpd.GeoDataFrame(
        pd.concat(
            [_lcf, _fp_gdf],
            ignore_index=True,
        ),
        geometry="geometry",
        crs=TARGET_CRS,
    )
    L_commercial_final["centroid"] = L_commercial_final.geometry.centroid
 
    _tn_dist = _fp_gdf["bezeich_TN"].value_counts().to_dict()
    print(f"\nFree parcel candidates: {len(_fp_gdf)}")
    print(f"  TN distribution:      {_tn_dist}")
    print(f"  was_retail=True:      {_fp_gdf['was_retail'].sum()}")
    print(f"\nL_commercial_final (all tracks): {len(L_commercial_final)}")
    print(f"  Buildings:   {(~L_commercial_final['is_greenfield']).sum()}")
    print(f"  Greenfield:  {L_commercial_final['is_greenfield'].sum()}")
    print(f"\nTier breakdown:\n{L_commercial_final['confidence_tier'].value_counts()}")
else:
    print("No free parcel candidates found within study area.")

In [ ]:
out_path = PATH_PROCESSED / "L_commercial_final.parquet"
L_commercial_final.drop(columns=["centroid"]).to_parquet(out_path, index=False)
print(f"Saved → {out_path}")
 
# Optional: also save as GeoPackage for QGIS inspection
gpkg_path = PATH_PROCESSED / "L_commercial_final.gpkg"
L_commercial_final.drop(columns=["centroid"]).to_file(
    gpkg_path, driver="GPKG", layer="L_commercial_final"
)
print(f"Saved → {gpkg_path}")

In [ ]:
L_commercial_final

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=300)
grid.to_crs(berlin.crs).plot(ax=ax, facecolor='lightgray', edgecolor='none', linewidth=0.5, label='Grid', alpha=0.5)
#grid_building_candidates[grid_building_candidates['n_blockers'] == 0].to_crs(berlin.crs).plot(ax=ax, facecolor='none', edgecolor='royalblue', linewidth=1, label='Gebaeude', alpha=0.5)
L_commercial_final.to_crs(berlin.crs).plot(ax=ax, facecolor='none', edgecolor='green', linewidth=1, label='Commercial Candidates', alpha=0.5)
#OSM_groud_floor_blockers[OSM_groud_floor_blockers['osm_type'] == 'way'].to_crs(berlin.crs).plot(ax=ax, facecolor='none', edgecolor='orange', linewidth=1, label='OSM Ground Floor Blockers', alpha=0.5)
#alkis_FS_TN_FNP.to_crs(berlin.crs).plot(ax=ax, facecolor='none', edgecolor='forestgreen', linewidth=1, label='ALKIS FS-TN-FNP', alpha=0.5)
berlin.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=2)

ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.2)

ax.legend()
plt.show()

In [ ]:
# ── CELL D-11 : Interactive HTML map of L_commercial_final ───────────────────
#
# Produces a Folium choropleth map showing all candidate building polygons
# coloured by confidence_tier, on an OSM background.
# Berlin boundary and inner-Ringbahn boundary are overlaid as outlines.
#
# Output: data/processed/L_commercial_final_map.html
# ─────────────────────────────────────────────────────────────────────────────

import folium
from folium.features import GeoJsonTooltip
import json, warnings

# ── 0. Reproject to WGS-84 (Folium requirement) ──────────────────────────────
L_map = (
    L_commercial_final
    .drop(columns=["centroid"], errors="ignore")
    .to_crs("EPSG:4326")
    .copy()
)

# Convert bool columns to plain Python bool so json.dumps doesn't choke
for col in L_map.select_dtypes(include=["bool"]).columns:
    L_map[col] = L_map[col].astype(object)

berlin_4326   = berlin.to_crs("EPSG:4326")
boundary_4326 = boundary.to_crs("EPSG:4326")

center_y = L_map.geometry.centroid.y.mean()
center_x = L_map.geometry.centroid.x.mean()

# ── 1. Style constants ────────────────────────────────────────────────────────
TIER_COLOR = {
    "A": "#c0392b",   # red       — former supermarket, physically pre-adapted
    "B": "#e67e22",   # amber     — former retail or high morphology score
    "C": "#2980b9",   # blue      — feasible, no prior history
}
TIER_LABEL = {
    "A": "A — Former supermarket",
    "B": "B — Former retail / high morphology",
    "C": "C — Feasible (no prior history)",
}

def _style(feature):
    tier  = feature["properties"].get("confidence_tier", "C")
    color = TIER_COLOR.get(tier, "#7f8c8d")
    return {
        "fillColor":   color,
        "color":       color,
        "weight":      0.6,
        "fillOpacity": 0.65,
    }

def _highlight(feature):
    return {"weight": 2.5, "fillOpacity": 0.90}

# ── 2. Simplify geometries for performance ────────────────────────────────────
# Convert to metric CRS → simplify at 1 m tolerance → back to WGS-84.
# This reduces vertex count on large Gründerzeit block polygons substantially
# (~30–60 % smaller GeoJSON) without any visible change at map zoom 12–14.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    L_map_simplified = (
        L_map
        .to_crs("EPSG:3035")
        .assign(geometry=lambda g: g.geometry.simplify(1.0, preserve_topology=True))
        .to_crs("EPSG:4326")
    )

# ── 3. Tooltip fields ─────────────────────────────────────────────────────────
_TOOLTIP_COLS = [
    "id", "confidence_tier", "gfk", "size_flag",
    "footprint_m2", "available_m2",
    "was_supermarket", "was_retail",
    "typ_klar", "brw_value", "morphology_score",
]
tooltip_cols = [c for c in _TOOLTIP_COLS if c in L_map_simplified.columns]

# Round floats for cleaner tooltips
for col in ["footprint_m2", "available_m2", "brw_value", "morphology_score"]:
    if col in L_map_simplified.columns:
        L_map_simplified[col] = L_map_simplified[col].round(1)

# ── 4. Build map ──────────────────────────────────────────────────────────────
m = folium.Map(
    location=[center_y, center_x],
    zoom_start=12,
    tiles="OpenStreetMap",
    prefer_canvas=True,
)

# ── 4a. Inner-Ringbahn boundary ───────────────────────────────────────────────
folium.GeoJson(
    data     = boundary_4326.__geo_interface__,
    name     = "Inner Ringbahn (study area)",
    style_function = lambda x: {
        "fillColor":   "none",
        "color":       "#8e44ad",
        "weight":      2.0,
        "dashArray":   "6 4",
        "fillOpacity": 0,
    },
    tooltip  = "Inner Ringbahn boundary",
).add_to(m)

# ── 4b. Berlin outer boundary ─────────────────────────────────────────────────
folium.GeoJson(
    data     = berlin_4326.__geo_interface__,
    name     = "Berlin boundary",
    style_function = lambda x: {
        "fillColor":   "none",
        "color":       "#2c3e50",
        "weight":      2.5,
        "fillOpacity": 0,
    },
    tooltip  = "Berlin boundary",
).add_to(m)

# ── 4c. Candidate building polygons, one layer per tier (enables toggle) ─────
for tier in ["A", "B", "C"]:
    subset = L_map_simplified[L_map_simplified["confidence_tier"] == tier].copy()
    if subset.empty:
        continue

    folium.GeoJson(
        data            = subset[tooltip_cols + ["geometry"]],
        name            = f"Tier {TIER_LABEL[tier]}",
        style_function  = _style,
        highlight_function = _highlight,
        tooltip         = GeoJsonTooltip(
            fields    = tooltip_cols,
            aliases   = [c.replace("_", " ").title() for c in tooltip_cols],
            localize  = True,
            sticky    = False,
        ),
    ).add_to(m)

# ── 4d. Layer control ─────────────────────────────────────────────────────────
folium.LayerControl(collapsed=False).add_to(m)

# ── 4e. Legend ────────────────────────────────────────────────────────────────
tier_counts = L_commercial_final["confidence_tier"].value_counts()

legend_rows = "".join(
    f"""
    <div style="display:flex; align-items:center; margin-bottom:5px;">
      <div style="width:16px; height:16px; background:{TIER_COLOR[t]};
                  opacity:0.75; border:1px solid #555; margin-right:8px;
                  flex-shrink:0;"></div>
      <span>{TIER_LABEL[t]} &nbsp;<b>({tier_counts.get(t, 0):,})</b></span>
    </div>"""
    for t in ["A", "B", "C"]
)

legend_html = f"""
<div style="
    position: fixed;
    bottom: 40px; left: 40px; z-index: 9999;
    background: rgba(255,255,255,0.93);
    padding: 12px 16px;
    border-radius: 8px;
    border: 1px solid #aaa;
    font-family: sans-serif;
    font-size: 12px;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.2);
    min-width: 260px;
">
  <div style="font-weight:bold; margin-bottom:8px; font-size:13px;">
    𝓛<sup>commercial</sup> — candidate supermarket locations
    &nbsp;<span style="font-weight:normal;">({len(L_commercial_final):,} buildings)</span>
  </div>
  {legend_rows}
  <hr style="border:none; border-top:1px solid #ddd; margin:8px 0;">
  <div style="color:#555; font-size:11px;">
    Polygon area = ALKIS building footprint.<br>
    Colour = confidence tier (A highest → C lowest).<br>
    Hover over polygon for attributes.
  </div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# ── 5. Save ───────────────────────────────────────────────────────────────────
out_html = PATH_PROCESSED / "L_commercial_final_map.html"
m.save(str(out_html))
print(f"Saved → {out_html}  ({out_html.stat().st_size / 1e6:.1f} MB)")

m   # renders inline in Jupyter

In [ ]:
if not GENERATE_PLOTS:
    import nbformat, pathlib

    _nb_path = pathlib.Path(__file__) if "__file__" in dir() else None
    # Fallback: set explicitly if auto-detection unavailable
    _nb_path = pathlib.Path("GEO_06_city_parcels.ipynb")  # ← set once per notebook

    _nb = nbformat.read(_nb_path, as_version=4)
    for _cell in _nb.cells:
        _cell["outputs"] = []
        _cell["execution_count"] = None
    nbformat.write(_nb, _nb_path)
    print(f"Outputs cleared: {_nb_path.name}")